# 03 — Salary plausibility model

Regress expected (log) pay on title, experience, education, and location, then persist the
fitted model **and the residual standard deviation** so Java can compute a z-score. A posting
whose promised pay is > 3σ above the fitted expectation for its role is flagged.

**Honest caveats (surfaced, not hidden):**
- `salary_range` is populated in only **~16%** of rows.
- It carries **no currency and no pay period** — bare `"min-max"` strings mixing hourly and
  annual pay. This inflates residuals; the model is a coarse outlier detector, not a wage oracle.
- Regressors `required_education`/`required_experience` are themselves ~40–45% missing.

In [ ]:
!pip -q install kagglehub

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
SEED = 42

import kagglehub
DATA_DIR = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")
csvs = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)
pref = [f for f in csvs if "fake_job_postings" in os.path.basename(f).lower()]
CSV = pref[0] if pref else csvs[0]
df = pd.read_csv(CSV)
print("Loaded:", CSV)
print("Shape:", df.shape)

In [ ]:
# --- Parse salary_range: bare "min-max", no currency/period ---
sal = df[df["salary_range"].notna()].copy()
parts = sal["salary_range"].astype(str).str.split("-", n=1, expand=True)
sal["lo"] = pd.to_numeric(parts[0], errors="coerce")
sal["hi"] = pd.to_numeric(parts[1], errors="coerce")
sal = sal.dropna(subset=["lo", "hi"])
sal = sal[(sal["lo"] > 0) & (sal["hi"] >= sal["lo"])]
sal["mid"] = (sal["lo"] + sal["hi"]) / 2.0
# Keep a plausible band to reduce the hourly/annual currency-unit chaos (documented limitation).
sal = sal[(sal["mid"] >= 1) & (sal["mid"] <= 1_000_000)]
sal["log_pay"] = np.log1p(sal["mid"])
print(f"Usable salaried rows: {len(sal):,}  ({len(sal)/len(df):.1%} of the dataset)")

In [ ]:
# --- Features ---
def country_of(loc):
    if not isinstance(loc, str) or not loc.strip():
        return "Unknown"
    return loc.split(",")[0].strip().upper()[:2] or "Unknown"

sal["country"] = sal["location"].apply(country_of)
for c in ["required_experience", "required_education", "employment_type"]:
    sal[c] = sal[c].fillna("Unknown").astype(str)
sal["title"] = sal["title"].fillna("").astype(str)

CAT_COLS = ["required_experience", "required_education", "employment_type", "country"]

In [ ]:
# --- Fit: OneHot(categoricals) + word TF-IDF(title) -> Ridge on log pay ---
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

pre = ColumnTransformer([
    ("cats", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ("title", TfidfVectorizer(max_features=300, ngram_range=(1, 1)), "title"),
])
salary_model = Pipeline([("pre", pre), ("reg", Ridge(alpha=1.0, random_state=SEED))])

Xtr, Xva, ytr, yva = train_test_split(
    sal[CAT_COLS + ["title"]], sal["log_pay"].values, test_size=0.2, random_state=SEED)
salary_model.fit(Xtr, ytr)

pred_va = salary_model.predict(Xva)
resid = yva - pred_va
resid_std = float(np.std(resid, ddof=1))
from sklearn.metrics import r2_score
print(f"R^2 (log pay, validation): {r2_score(yva, pred_va):.3f}  <- expected LOW; salary_range is noisy")
print(f"Residual std (log space):  {resid_std:.4f}   <- Java uses this for the z-score")

In [ ]:
# --- Persist model + residual std + feature spec ---
import joblib
joblib.dump(salary_model, os.path.join(OUT, "salary_model.joblib"))
meta = {
    "target": "log1p(midpoint of salary_range)",
    "residual_std_log": resid_std,
    "z_flag_threshold": 3.0,
    "cat_cols": CAT_COLS,
    "title_tfidf_max_features": 300,
    "note": "z = (log1p(promised_mid) - predicted_log) / residual_std_log; flag if z > 3.",
}
with open(os.path.join(OUT, "salary_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("Saved salary_model.joblib and salary_meta.json to", OUT)
print(json.dumps(meta, indent=2))

A "data entry" role promising far more than its fitted expectation is an **arithmetic**
signal, not a text one — which is exactly why it is a separate model. Given the data's
noise, treat it as a coarse flag that contributes to the verdict, never as a standalone verdict.